# MAWRID Benchmark — 3-Stage Pipeline
**Stage 1** Vision → raw OCR text | **Stage 2** Text → doc type | **Stage 3** Text → fields

Models: **Claude** (ground truth) · **Groq** (llama-4-scout) · **Gemma3** (local HF)

Upload to `/gdrive/MyDrive/mawrid_data/`: `schema_v2.json` and `raw_pdfs/` folder.

## 1 — Installs

In [ ]:
!pip install pdf2image==1.17.0 "pillow==12.1.0" --force-reinstall -q
!apt-get install -y poppler-utils -q
!pip install groq anthropic -q
!pip install transformers accelerate bitsandbytes -q
!pip install qwen-vl-utils -q

## 2 — Mount Drive & Keys

In [ ]:
import os, json, time, base64, re
from os.path import join
from glob import glob
from tqdm.auto import tqdm
from difflib import SequenceMatcher
from pdf2image import convert_from_path
from PIL import Image, ImageEnhance
from google.colab import userdata, drive

drive.mount('/gdrive')

data_dir      = "/gdrive/MyDrive/mawrid_data"
GT_PATH       = f"{data_dir}/ground_truth.json"
HUMAN_GT_PATH = f"{data_dir}/human_gt.json"

os.environ["GROQ_API_KEY"]      = userdata.get('groq-key')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('anthropic-key')

print("Keys loaded ✓")

## 3 — Load Schema

In [ ]:
with open(f"{data_dir}/schema_v2.json", encoding="utf-8") as f:
    SCHEMA = json.load(f)

DOCUMENTS = SCHEMA["documents"]

DOC_TYPES_LIST = "\n".join(
    f"  {key} — {doc['label_ar']}"
    for key, doc in DOCUMENTS.items()
)

SCHEMA_FIELDS = {
    key: [f["label_ar"] for f in doc.get("fields", [])]
    for key, doc in DOCUMENTS.items()
}

# field name → type hint for the extraction template
SCHEMA_FIELD_TYPES = {
    key: {f["label_ar"]: f.get("type", "text") for f in doc.get("fields", [])}
    for key, doc in DOCUMENTS.items()
}

print(f"Schema loaded ✓  ({len(DOCUMENTS)} document types)")

## 4 — PDF → Images

In [ ]:
def preprocess_image(image, maxwidth=800):
    gray = image.convert('L')
    if gray.width > maxwidth:
        h = int(gray.height * maxwidth / gray.width)
        gray = gray.resize((maxwidth, h), Image.LANCZOS)
    return ImageEnhance.Contrast(gray).enhance(1.5)

def convert_pdf_to_images(pdf_path, output_dir, maxwidth=800):
    name = os.path.basename(pdf_path).replace('.pdf', '')
    out  = join(output_dir, name)
    os.makedirs(out, exist_ok=True)
    paths = []
    for i, img in enumerate(convert_from_path(pdf_path, dpi=200), 1):
        p = join(out, f"{name}_page_{i:03d}.jpeg")
        preprocess_image(img, maxwidth).save(p, 'JPEG', quality=85, optimize=True)
        paths.append(p)
    return paths

# pdf_files = glob(f"{data_dir}/raw_pdfs/*.pdf")
# if pdf_files:
#     for pdf in tqdm(pdf_files, desc="Converting PDFs"):
#         convert_pdf_to_images(pdf, f"{data_dir}/processed_images")
#     print(f"Done — {len(pdf_files)} PDFs")
# else:
#     print("No PDFs found — skipping")

## 5 — Prompts

In [ ]:
SYSTEM_OCR = (
    "You are an OCR engine that transcribes Arabic official documents exactly as written — "
    "no paraphrasing, no translation, no summarization."
)

PROMPT_OCR = """Extract ALL visible text from this document image exactly as it appears.

Rules:
- Arabic stays Arabic, English stays English — do NOT romanize or translate
- Copy letter by letter — dots matter (ي/ن, ز/ر, ث/ت/ب)
- Include headers, body, footer, stamps, handwritten annotations, all reference numbers and dates
- If illegible write [unclear]

Return plain text only."""

_CLASSIFY_TEMPLATE = """You are given the raw OCR text of an Arabic official document.
Identify the document type. Reply with ONLY a JSON object:
{{"doc_type": "<key>", "confidence": 0.0}}

You MUST select doc_type from ONLY these exact keys — do NOT invent, modify, or translate them:
{doc_types_list}

CRITICAL: If unsure, pick the closest matching key from the list above. Never output a key that is not in the list.

OCR TEXT:
{raw_text}"""

def build_classify_prompt(raw_text: str) -> str:
    return _CLASSIFY_TEMPLATE.format(
        doc_types_list=DOC_TYPES_LIST,
        raw_text=raw_text
    )

_EXTRACT_TEMPLATE = """Extract fields from this Arabic document by filling in the JSON template below.

Document type: {doc_type}

JSON template to fill (replace each placeholder with the value from the text, keep null if not found):
{json_template}

Rules:
- Output ONLY the filled JSON — no extra keys, no explanation
- Do NOT rename or add any keys
- Return ALL keys from the template above — use null for any field not found in the text
- For date placeholders (DD/MM/YYYY): always return numeric format like 28/06/2000, never words
- Values must come from the OCR text below

OCR text:
{raw_text}"""

def _field_placeholder(field_name: str, field_type: str) -> object:
    if field_type == "date" or "تاريخ" in field_name:
        return "DD/MM/YYYY"
    if field_type == "number":
        return 0
    return None

def build_extract_prompt(raw_text: str, doc_type: str) -> str:
    expected    = SCHEMA_FIELDS.get(doc_type, [])
    field_types = SCHEMA_FIELD_TYPES.get(doc_type, {})
    if expected:
        template = json.dumps(
            {f: _field_placeholder(f, field_types.get(f, "text")) for f in expected},
            ensure_ascii=False, indent=2
        )
    else:
        template = json.dumps({"الحقول": "استخرج كل الحقول الموجودة"}, ensure_ascii=False)
    return _EXTRACT_TEMPLATE.format(
        doc_type=doc_type, json_template=template, raw_text=raw_text
    )

print("Prompts loaded ✓")

## 6 — Load Gemma3 Model (needs GPU)

In [ ]:
from google.colab import userdata
hf_token = userdata.get('my-hf')

!hf auth login --token {hf_token}

In [ ]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

GEMMA_MODEL_ID = "google/gemma-3-4b-it"

gemma_model = Gemma3ForConditionalGeneration.from_pretrained(
    GEMMA_MODEL_ID, device_map="auto", torch_dtype="auto"
).eval()

gemma_processor = AutoProcessor.from_pretrained(GEMMA_MODEL_ID)

print(f"Gemma3 loaded ✓  (device: {next(gemma_model.parameters()).device})")

## 7 — Stage Functions (Claude · Groq · Gemma3)

In [ ]:
STAGE_FNS = {
    "claude": (stage1_claude, stage2_claude, stage3_claude),
    "groq":   (stage1_groq,   stage2_groq,   stage3_groq),
    "gemma":  (stage1_gemma,  stage2_gemma,  stage3_gemma),
    "qwen":   (stage1_qwen,   stage2_qwen,   stage3_qwen),
}

def run_pipeline(image_paths: list, doc_id: str, provider: str) -> dict:
    s1, s2, s3 = STAGE_FNS[provider]

    pages_text, t1_total = [], 0.0
    for path in image_paths:
        text, t = s1(path)
        pages_text.append(text)
        t1_total += t
    raw_text = "\n\n--- PAGE BREAK ---\n\n".join(pages_text)

    cls_result, t2 = s2(raw_text)
    doc_type        = cls_result.get("doc_type", "unknown")

    fields, t3 = s3(raw_text, doc_type)

    return {
        "doc_id":     doc_id,
        "provider":   provider,
        "raw_text":   raw_text,
        "doc_type":   doc_type,
        "confidence": cls_result.get("confidence"),
        "fields":     fields,
        "timing": {
            "stage1": round(t1_total, 2),
            "stage2": round(t2, 2),
            "stage3": round(t3, 2),
            "total":  round(t1_total + t2 + t3, 2),
        },
    }

print("Pipeline runner loaded ✓")

## 8 — Pipeline Runner

In [ ]:
STAGE_FNS = {
    "claude": (stage1_claude, stage2_claude, stage3_claude),
    "groq":   (stage1_groq,   stage2_groq,   stage3_groq),
    "gemma":  (stage1_gemma,  stage2_gemma,  stage3_gemma),
}

def run_pipeline(image_paths: list, doc_id: str, provider: str) -> dict:
    s1, s2, s3 = STAGE_FNS[provider]

    pages_text, t1_total = [], 0.0
    for path in image_paths:
        text, t = s1(path)
        pages_text.append(text)
        t1_total += t
    raw_text = "\n\n--- PAGE BREAK ---\n\n".join(pages_text)

    cls_result, t2 = s2(raw_text)
    doc_type        = cls_result.get("doc_type", "unknown")

    fields, t3 = s3(raw_text, doc_type)

    return {
        "doc_id":     doc_id,
        "provider":   provider,
        "raw_text":   raw_text,
        "doc_type":   doc_type,
        "confidence": cls_result.get("confidence"),
        "fields":     fields,
        "timing": {
            "stage1": round(t1_total, 2),
            "stage2": round(t2, 2),
            "stage3": round(t3, 2),
            "total":  round(t1_total + t2 + t3, 2),
        },
    }

print("Pipeline runner loaded ✓")

## 9 — Smoke Test (one image, all 3 models)
Run this **before** generating ground truth to verify all providers work correctly.

In [ ]:
SMOKE_IMAGE  = "/gdrive/MyDrive/mawrid_data/processed_images/0001/0001_page_001.jpeg"
smoke_doc_id = os.path.basename(os.path.dirname(SMOKE_IMAGE))   # → "0001"
smoke_results = {}
print(f"Smoke image : {SMOKE_IMAGE}")
print(f"Smoke doc   : {smoke_doc_id}")

In [ ]:
# ── Claude ────────────────────────────────────────────────────────────────────
try:
    result = run_pipeline([SMOKE_IMAGE], smoke_doc_id, provider="claude")
    smoke_results["claude"] = result
    print(f"Stage 1 OCR ({result['timing']['stage1']}s):\n{result['raw_text'][:400]}")
    print(f"\nStage 2 Doc Type ({result['timing']['stage2']}s): {result['doc_type']}")
    print(f"Stage 3 Fields ({result['timing']['stage3']}s):")
    for k, v in result["fields"].items():
        print(f"  {k}: {v}")
except Exception as e:
    import traceback; traceback.print_exc()
    smoke_results["claude"] = None

In [ ]:
# ── Groq ──────────────────────────────────────────────────────────────────────
try:
    result = run_pipeline([SMOKE_IMAGE], smoke_doc_id, provider="groq")
    smoke_results["groq"] = result
    print(f"Stage 1 OCR ({result['timing']['stage1']}s):\n{result['raw_text'][:400]}")
    print(f"\nStage 2 Doc Type ({result['timing']['stage2']}s): {result['doc_type']}")
    print(f"Stage 3 Fields ({result['timing']['stage3']}s):")
    for k, v in result["fields"].items():
        print(f"  {k}: {v}")
except Exception as e:
    import traceback; traceback.print_exc()
    smoke_results["groq"] = None

In [ ]:
# ── Gemma3 ────────────────────────────────────────────────────────────────────
try:
    result = run_pipeline([SMOKE_IMAGE], smoke_doc_id, provider="gemma")
    smoke_results["gemma"] = result
    print(f"Stage 1 OCR ({result['timing']['stage1']}s):\n{result['raw_text'][:400]}")
    print(f"\nStage 2 Doc Type ({result['timing']['stage2']}s): {result['doc_type']}")
    print(f"Stage 3 Fields ({result['timing']['stage3']}s):")
    for k, v in result["fields"].items():
        print(f"  {k}: {v}")
except Exception as e:
    import traceback; traceback.print_exc()
    smoke_results["gemma"] = None

In [ ]:
# ── Qwen2.5-VL ────────────────────────────────────────────────────────────────
try:
    result = run_pipeline([SMOKE_IMAGE], smoke_doc_id, provider="qwen")
    smoke_results["qwen"] = result
    print(f"Stage 1 OCR ({result['timing']['stage1']}s):\n{result['raw_text'][:400]}")
    print(f"\nStage 2 Doc Type ({result['timing']['stage2']}s): {result['doc_type']}")
    print(f"Stage 3 Fields ({result['timing']['stage3']}s):")
    for k, v in result["fields"].items():
        print(f"  {k}: {v}")
except Exception as e:
    import traceback; traceback.print_exc()
    smoke_results["qwen"] = None

In [ ]:
# ── Summary + Save ────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("  Smoke Test Summary")
print(f"{'='*55}")
print(f"{'Provider':<12} {'Doc Type':<30} {'Fields':>6} {'Time':>7}")
print("-" * 55)
for p, r in smoke_results.items():
    if r:
        print(f"{p:<12} {r['doc_type']:<30} {len(r['fields']):>6} {r['timing']['total']:>6.1f}s")
    else:
        print(f"{p:<12} {'FAILED':<30}")

SMOKE_PATH = f"{data_dir}/smoke_test_result.json"
save_data = {
    "image":   SMOKE_IMAGE,
    "doc_id":  smoke_doc_id,
    "run_at":  time.strftime("%Y-%m-%d %H:%M:%S"),
    "providers": {}
}
for p, r in smoke_results.items():
    if r:
        save_data["providers"][p] = {
            "doc_type":      r["doc_type"],
            "confidence":    r["confidence"],
            "timing":        r["timing"],
            "stage1_ocr":    r["raw_text"],
            "stage2_cls":    r["doc_type"],
            "stage3_fields": r["fields"],
        }
    else:
        save_data["providers"][p] = {"error": "FAILED"}

with open(SMOKE_PATH, "w", encoding="utf-8") as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)
print(f"\nFull results saved → {SMOKE_PATH}")

In [ ]:
def generate_ground_truth(num_docs: int = 15):
    image_root = f"{data_dir}/processed_images"
    doc_dirs   = sorted(glob(f"{image_root}/*/"))[:num_docs]

    gt = {}
    if os.path.exists(GT_PATH):
        with open(GT_PATH, encoding="utf-8") as f:
            gt = json.load(f)
        print(f"Resuming — {len(gt)} docs already done")

    for doc_dir in tqdm(doc_dirs, desc="Claude Ground Truth"):
        doc_id = os.path.basename(doc_dir.rstrip("/"))
        if doc_id in gt:
            print(f"  skip {doc_id}")
            continue

        pages  = sorted(glob(f"{doc_dir}*.jpeg"))
        result = run_pipeline(pages, doc_id, provider="claude")

        gt[doc_id] = {
            "doc_type":       result["doc_type"],
            "reference_text": result["raw_text"],
            "fields":         result["fields"],
            "num_pages":      len(pages),
            "claude_model":   "claude-sonnet-4-6",
            "generated_at":   "2026-05-19",
        }
        with open(GT_PATH, "w", encoding="utf-8") as f:
            json.dump(gt, f, ensure_ascii=False, indent=2)

        t = result["timing"]["total"]
        print(f"  ✓ {doc_id}: {result['doc_type']} | {len(pages)}p | {t:.1f}s")

    print(f"\nGround truth ready → {GT_PATH}  ({len(gt)} docs)")
    return gt


# Uncomment, run once, then recomment:
# gt = generate_ground_truth(num_docs=15)

## 11 — Scorer

In [ ]:
def normalize_arabic(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)  # tashkeel
    text = re.sub(r'[أإآ]', 'ا', text)                  # alef
    text = text.replace('ة', 'ه').replace('ى', 'ي')     # ta marbuta + ya
    text = text.replace('_', ' ')                        # schema underscores
    return text.strip()

def score_doc(gt_doc: dict, pred: dict) -> dict:
    gt_f   = {normalize_arabic(k): normalize_arabic(str(v))
               for k, v in gt_doc.get("fields", {}).items() if v is not None}
    pred_f = {normalize_arabic(k): normalize_arabic(str(v))
               for k, v in pred.get("fields", {}).items() if v is not None}

    matched   = set(gt_f) & set(pred_f)
    precision = len(matched) / len(pred_f) if pred_f else 0.0
    recall    = len(matched) / len(gt_f)   if gt_f   else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall > 0 else 0.0)

    correct_vals = sum(
        1 for k in matched
        if SequenceMatcher(None, gt_f[k], pred_f[k]).ratio() >= 0.85
    )
    val_acc = correct_vals / len(matched) if matched else 0.0

    return {
        "cls_correct":    int(normalize_arabic(pred.get("doc_type", "")) ==
                              normalize_arabic(gt_doc.get("doc_type", ""))),
        "precision":      round(precision, 3),
        "recall":         round(recall, 3),
        "f1":             round(f1, 3),
        "value_accuracy": round(val_acc, 3),
        "matched_keys":   len(matched),
    }

print("Scorer loaded ✓")

## 12 — Benchmark Runner

In [ ]:
from IPython.display import display, HTML

def view_ground_truth(gt_path: str = HUMAN_GT_PATH, ocr_preview_chars: int = 400):
    with open(gt_path, encoding="utf-8") as f:
        gt = json.load(f)

    cards = []
    for doc_id, doc in sorted(gt.items()):
        ocr_preview = doc.get("reference_text", "")[:ocr_preview_chars].replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
        if len(doc.get("reference_text", "")) > ocr_preview_chars:
            ocr_preview += "<br><em>…</em>"

        ocr_section = (
            f"""<div style='padding:12px 16px;border-bottom:1px solid #eee'>
            <span style='font-size:11px;color:#888;text-transform:uppercase'>المرحلة 1 — نص OCR</span><br>
            <div style='margin-top:6px;font-size:13px;line-height:1.6;color:#333;background:#fff;padding:8px;border-radius:4px;border:1px solid #eee;max-height:180px;overflow-y:auto'>
              {ocr_preview}
            </div>
          </div>"""
            if doc.get("reference_text")
            else ""
        )

        fields_rows = "".join(
            f"<tr><td style='padding:4px 10px;border-bottom:1px solid #eee;color:#555'>{k}</td>"
            f"<td style='padding:4px 10px;border-bottom:1px solid #eee;font-weight:500'>"
            f"{'<span style=\"color:#aaa\">null</span>' if v is None else v}</td></tr>"
            for k, v in doc.get("fields", {}).items()
        )

        doc_label = DOCUMENTS.get(doc["doc_type"], {}).get("label_ar", doc["doc_type"])
        num_pages  = doc.get("num_pages", "—")

        cards.append(f"""
        <div style='border:1px solid #ddd;border-radius:8px;margin:16px 0;font-family:Arial;direction:rtl'>
          <div style='background:#2c3e50;color:white;padding:10px 16px;border-radius:7px 7px 0 0;display:flex;justify-content:space-between;align-items:center'>
            <span style='font-size:16px;font-weight:bold'>وثيقة {doc_id}</span>
            <span style='font-size:13px;opacity:.8'>{num_pages} صفحة</span>
          </div>

          <div style='padding:12px 16px;background:#f9f9f9;border-bottom:1px solid #eee'>
            <span style='font-size:11px;color:#888;text-transform:uppercase'>المرحلة 2 — نوع الوثيقة</span><br>
            <span style='font-size:15px;font-weight:bold;color:#2980b9'>{doc["doc_type"]}</span>
            <span style='color:#888;margin-right:8px'>({doc_label})</span>
          </div>

          {ocr_section}

          <div style='padding:12px 16px'>
            <span style='font-size:11px;color:#888;text-transform:uppercase'>المرحلة 3 — الحقول المستخرجة</span><br>
            <table style='width:100%;margin-top:8px;border-collapse:collapse;font-size:13px'>
              {fields_rows if fields_rows else "<tr><td style='color:#aaa'>لا توجد حقول</td></tr>"}
            </table>
          </div>
        </div>
        """)

    display(HTML(f"<div style='max-width:900px'>{''.join(cards)}</div>"))
    print(f"✓ {len(gt)} وثيقة  ({os.path.basename(gt_path)})")

view_ground_truth()

## ~~10c — Human Ground Truth Editor~~ ✅ Done
Human GT saved to `human_gt.json` — editor no longer needed.

In [ ]:
# Human GT already saved — editor skipped.
print(f"Human GT exists: {os.path.exists(HUMAN_GT_PATH)}")

In [ ]:
def run_benchmark(provider: str, num_docs: int = 15, max_pages: int = 3) -> list:
    if not os.path.exists(HUMAN_GT_PATH):
        raise FileNotFoundError("Run the Human Ground Truth Editor first and save human_gt.json.")
    with open(HUMAN_GT_PATH, encoding="utf-8") as f:
        gt = json.load(f)

    image_root = f"{data_dir}/processed_images"
    results    = []

    for doc_id in tqdm(sorted(gt.keys())[:num_docs], desc=f"[{provider}]"):
        pages = sorted(glob(f"{image_root}/{doc_id}/*.jpeg"))
        if max_pages and len(pages) > max_pages:
            pages = pages[:max_pages]
        pred   = run_pipeline(pages, doc_id, provider=provider)
        scores = score_doc(gt[doc_id], pred)
        results.append({
            "doc_id":    doc_id,
            "provider":  provider,
            "gt_type":   gt[doc_id]["doc_type"],
            "pred_type": pred["doc_type"],
            **scores,
            "timing":    pred["timing"],
        })

    return results


def print_summary(results: list):
    n, p = len(results), results[0]["provider"]
    cls  = sum(r["cls_correct"] for r in results)
    f1   = sum(r["f1"] for r in results) / n
    va   = sum(r["value_accuracy"] for r in results) / n
    avgt = sum(r["timing"]["total"] for r in results) / n

    print(f"\n{'='*65}")
    print(f"  {p.upper():<12} Cls={cls}/{n} ({cls/n:.0%})  F1={f1:.3f}  VA={va:.3f}  Avg={avgt:.1f}s")
    print(f"{'='*65}")
    for r in results:
        icon = "✓" if r["cls_correct"] else "✗"
        t    = r["timing"]
        print(f"  {icon} {r['doc_id']}  {r['pred_type']:<28}  "
              f"F1={r['f1']:.2f} VA={r['value_accuracy']:.2f}  "
              f"[{t['stage1']:.1f}+{t['stage2']:.1f}+{t['stage3']:.1f}]s")


def save_results(results: list, provider: str) -> str:
    n    = len(results)
    cls  = sum(r["cls_correct"] for r in results)
    f1   = sum(r["f1"] for r in results) / n
    va   = sum(r["value_accuracy"] for r in results) / n
    avgt = sum(r["timing"]["total"] for r in results) / n
    ts   = time.strftime("%Y%m%d_%H%M%S")

    per_doc_table = []
    for r in results:
        t = r["timing"]
        per_doc_table.append({
            "doc_id":         r["doc_id"],
            "gt_type":        r["gt_type"],
            "pred_type":      r["pred_type"],
            "cls_correct":    bool(r["cls_correct"]),
            "f1":             r["f1"],
            "value_accuracy": r["value_accuracy"],
            "timing": {
                "stage1_ocr_s":      t["stage1"],
                "stage2_classify_s": t["stage2"],
                "stage3_extract_s":  t["stage3"],
                "total_s":           t["total"],
            },
        })

    payload = {
        "provider": provider,
        "run_at":   ts,
        "summary": {
            "cls_correct":        cls,
            "total_docs":         n,
            "cls_accuracy":       round(cls / n, 3),
            "avg_f1":             round(f1, 3),
            "avg_value_accuracy": round(va, 3),
            "avg_time_s":         round(avgt, 1),
        },
        "per_doc": per_doc_table,
    }

    path = f"{data_dir}/results_{provider}_{ts}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Saved → {path}")
    return path


def comparison_table(*result_lists):
    header = f"{'Provider':<12} {'Cls Acc':>8} {'Field F1':>9} {'Val Acc':>8} {'Avg Time':>9}"
    print("\n" + "Ground Truth: human_gt.json")
    print(header)
    print("-" * len(header))
    for results in result_lists:
        if not results:
            continue
        n, p = len(results), results[0]["provider"]
        cls  = sum(r["cls_correct"] for r in results)
        f1   = sum(r["f1"] for r in results) / n
        va   = sum(r["value_accuracy"] for r in results) / n
        avgt = sum(r["timing"]["total"] for r in results) / n
        print(f"{p:<12} {cls}/{n:>5}   {f1:>8.3f} {va:>8.3f} {avgt:>8.1f}s")

print("Benchmark runner loaded ✓")

In [ ]:
# ── Qwen2.5-VL ────────────────────────────────────────────────────────────────
qwen_results = run_benchmark(provider="qwen", num_docs=15)
print_summary(qwen_results)
save_results(qwen_results, "qwen")

# ── Final comparison table: Claude vs Groq vs Gemma3 vs Qwen2.5-VL ───────────
comparison_table(claude_results, groq_results, gemma_results, qwen_results)

## 6b — Load Qwen2.5-VL-7B
Run this **after** the Gemma benchmark — it frees Gemma's VRAM first.

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# Free Gemma VRAM before loading Qwen
del gemma_model
torch.cuda.empty_cache()

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

_qwen_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID, device_map="auto", quantization_config=_qwen_bnb
).eval()

qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)

# Register Qwen in the pipeline now that the model is loaded
STAGE_FNS["qwen"] = (stage1_qwen, stage2_qwen, stage3_qwen)

print(f"Qwen2.5-VL loaded ✓  (device: {next(qwen_model.parameters()).device})")

In [ ]:
# ── Groq ──────────────────────────────────────────────────────────────────────
groq_results = run_benchmark(provider="groq", num_docs=15)
print_summary(groq_results)
save_results(groq_results, "groq")

In [ ]:
# ── Gemma3 ────────────────────────────────────────────────────────────────────
gemma_results = run_benchmark(provider="gemma", num_docs=15)
print_summary(gemma_results)
save_results(gemma_results, "gemma")

In [ ]:
# ── Claude — run live for real timing ────────────────────────────────────────
claude_results = run_benchmark(provider="claude", num_docs=15)
print_summary(claude_results)
save_results(claude_results, "claude")

In [ ]:
# ── Final comparison table: Claude vs Groq vs Gemma3 ─────────────────────────
comparison_table(claude_results, groq_results, gemma_results)